In [2]:
!pip -q install -U transformers datasets evaluate accelerate sentencepiece rouge_score sacrebleu safetensors

from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 145.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 54.6 MB/s eta 0:00:00
Mounted at /content/drive


In [3]:
import os
import gc
import re
import json
import random
import inspect
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PROJECT_DIR = Path("/content/drive/MyDrive/Senior2_Medical_Captioning")

MASTER_PATH = PROJECT_DIR / "master_caption_concepts_scored.csv"
MAPPING_PATH = PROJECT_DIR / "cui_to_umls_terms.csv"

OUTPUT_DIR = PROJECT_DIR / "models" / "exp4_mixed_gt_pred_micro_terms_t5_base_safe5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "t5-base"

MIX_GT_RATIO = 0.50
VALID_SAMPLE_SIZE = 3000

FAST_DEBUG = False
TRAIN_DEBUG_SIZE = 20000
VALID_DEBUG_SIZE = 1000

print("PROJECT_DIR:", PROJECT_DIR)
print("MASTER_PATH:", MASTER_PATH)
print("MAPPING_PATH:", MAPPING_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MIX_GT_RATIO:", MIX_GT_RATIO)
print("FAST_DEBUG:", FAST_DEBUG)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: CPU only")

PROJECT_DIR: /content/drive/MyDrive/Senior2_Medical_Captioning
MASTER_PATH: /content/drive/MyDrive/Senior2_Medical_Captioning/master_caption_concepts_scored.csv
MAPPING_PATH: /content/drive/MyDrive/Senior2_Medical_Captioning/cui_to_umls_terms.csv
OUTPUT_DIR: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5
MIX_GT_RATIO: 0.5
FAST_DEBUG: False
GPU: NVIDIA A100-SXM4-80GB


In [4]:
!nvidia-smi

Thu Jul  9 19:09:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   38C    P0             54W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [5]:
if not MASTER_PATH.exists():
    raise FileNotFoundError(f"MASTER_PATH not found:\n{MASTER_PATH}")

master_df = pd.read_csv(MASTER_PATH)

print("Master shape:", master_df.shape)
print("Master columns:")
print(master_df.columns.tolist())

display(master_df.head())

LABEL_VOCAB_PATHS = [
    Path("/content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json"),
    PROJECT_DIR / "label_vocab.json",
]

LABEL_VOCAB_PATH = None

for p in LABEL_VOCAB_PATHS:
    if p.exists():
        LABEL_VOCAB_PATH = p
        break

if LABEL_VOCAB_PATH is None:
    raise FileNotFoundError("label_vocab.json not found in expected paths.")

print("\nUsing LABEL_VOCAB_PATH:", LABEL_VOCAB_PATH)

with open(LABEL_VOCAB_PATH, "r", encoding="utf-8") as f:
    label_obj = json.load(f)

if isinstance(label_obj, dict) and "labels" in label_obj:
    cui_list = label_obj["labels"]
elif isinstance(label_obj, list):
    cui_list = label_obj
else:
    raise ValueError("Unsupported label_vocab structure.")

cui_list = [str(c) for c in cui_list]

print("Number of CUIs:", len(cui_list))
print("First 10 CUIs:", cui_list[:10])

Master shape: (116604, 20)
Master columns:
['ID', 'reference_caption', 'License', 'Attribution', 'gt_CUIs', 'pred_CUIs_micro', 'pred_CUIs_coverage', 'split', 'image_filename', 'caption_image_zip_path', 'concept_image_zip_path', 'gt_CUIs_count', 'pred_CUIs_micro_count', 'pred_CUIs_coverage_count', 'pred_CUIs_micro_precision', 'pred_CUIs_micro_recall', 'pred_CUIs_micro_f1', 'pred_CUIs_coverage_precision', 'pred_CUIs_coverage_recall', 'pred_CUIs_coverage_f1']


,ID,reference_caption,License,Attribution,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage,split,image_filename,caption_image_zip_path,concept_image_zip_path,gt_CUIs_count,pred_CUIs_micro_count,pred_CUIs_coverage_count,pred_CUIs_micro_precision,pred_CUIs_micro_recall,pred_CUIs_micro_f1,pred_CUIs_coverage_precision,pred_CUIs_coverage_recall,pred_CUIs_coverage_f1
0,ImageCLEFmedical_Caption_2026_train_0,Head CT demonstrating left parotiditis.,CC BY,Peres et al.,C0040405,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_0.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,1,1,1,1.0,1.0,1.000000,1.0,1.0,1.000000
1,ImageCLEFmedical_Caption_2026_train_1,Chest X-ray showing enlarged cardiac silhouett...,CC BY-NC,Al Mulhim et al.,C1306645;C0817096;C0442800;C0018787;C0242073,C0817096;C1306645,C0817096;C1306645,train,ImageCLEFmedical_Caption_2026_train_1.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,5,2,2,1.0,0.4,0.571429,1.0,0.4,0.571429
2,ImageCLEFmedical_Caption_2026_train_2,CT chest axial view showing a huge ascending a...,CC BY-NC,Al Mulhim et al.,C0040405;C0856747,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_2.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.5,0.666667,1.0,0.5,0.666667
3,ImageCLEFmedical_Caption_2026_train_3,Acquired renal cysts in end-stage renal failur...,CC BY,Vester et al.,C0041618,C0041618,C0041618,train,ImageCLEFmedical_Caption_2026_train_3.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,1,1,1,1.0,1.0,1.000000,1.0,1.0,1.000000
4,ImageCLEFmedical_Caption_2026_train_4,Computed tomography (CT) shows floating thromb...,CC BY,Sato et al.,C0040405;C0040053,C0040405,C0040405,train,ImageCLEFmedical_Caption_2026_train_4.jpg,dev_caption/images/ImageCLEFmedical_Caption_20...,dev_concept/images/ImageCLEFmedical_Caption_20...,2,1,1,1.0,0.5,0.666667,1.0,0.5,0.666667



Using LABEL_VOCAB_PATH: /content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json
Number of CUIs: 2646
First 10 CUIs: ['C0000726', 'C0000741', 'C0000833', 'C0000846', 'C0000962', 'C0001074', 'C0001080', 'C0001162', 'C0001168', 'C0001208']


In [6]:
def pick_column(df, candidates, target_name, required=True):
    lower_map = {c.lower(): c for c in df.columns}

    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    for col in df.columns:
        col_l = col.lower()
        for cand in candidates:
            if cand.lower() in col_l:
                return col

    if required:
        raise ValueError(
            f"Could not find column for {target_name}. "
            f"Available columns: {df.columns.tolist()}"
        )

    return None


id_col = pick_column(master_df, ["ID", "image_id", "ImageID", "sample_id"], "ID")
split_col = pick_column(master_df, ["split", "Split", "data_split", "subset"], "split")
caption_col = pick_column(master_df, ["reference_caption", "caption", "target_text", "reference"], "reference_caption")

gt_col = pick_column(master_df, ["gt_CUIs", "gt_cuis", "ground_truth_cuis", "gt_concepts", "concepts"], "gt_CUIs")

pred_micro_col = pick_column(
    master_df,
    ["pred_CUIs_micro", "pred_cuis_micro", "pred_micro_cuis", "micro_CUIs", "micro_cuis"],
    "pred_CUIs_micro"
)

pred_coverage_col = pick_column(
    master_df,
    ["pred_CUIs_coverage", "pred_cuis_coverage", "pred_coverage_cuis", "coverage_CUIs", "coverage_cuis"],
    "pred_CUIs_coverage"
)

print("Detected columns:")
print("ID:", id_col)
print("split:", split_col)
print("reference_caption:", caption_col)
print("gt_CUIs:", gt_col)
print("pred_CUIs_micro:", pred_micro_col)
print("pred_CUIs_coverage:", pred_coverage_col)

std_df = master_df.copy()

std_df["ID"] = std_df[id_col].astype(str)
std_df["split"] = std_df[split_col].astype(str).str.lower().str.strip()
std_df["reference_caption"] = std_df[caption_col].astype(str)
std_df["gt_CUIs"] = std_df[gt_col].fillna("").astype(str)
std_df["pred_CUIs_micro"] = std_df[pred_micro_col].fillna("").astype(str)
std_df["pred_CUIs_coverage"] = std_df[pred_coverage_col].fillna("").astype(str)

split_map = {
    "val": "valid",
    "validation": "valid",
    "dev": "valid",
    "training": "train"
}

std_df["split"] = std_df["split"].replace(split_map)

master_df = std_df

print("\nSplit counts:")
print(master_df["split"].value_counts())

display(master_df[[
    "ID",
    "split",
    "reference_caption",
    "gt_CUIs",
    "pred_CUIs_micro",
    "pred_CUIs_coverage"
]].head())

Detected columns:
ID: ID
split: split
reference_caption: reference_caption
gt_CUIs: gt_CUIs
pred_CUIs_micro: pred_CUIs_micro
pred_CUIs_coverage: pred_CUIs_coverage

Split counts:
split
train    97364
valid    19240
Name: count, dtype: int64


,ID,split,reference_caption,gt_CUIs,pred_CUIs_micro,pred_CUIs_coverage
0,ImageCLEFmedical_Caption_2026_train_0,train,Head CT demonstrating left parotiditis.,C0040405,C0040405,C0040405
1,ImageCLEFmedical_Caption_2026_train_1,train,Chest X-ray showing enlarged cardiac silhouett...,C1306645;C0817096;C0442800;C0018787;C0242073,C0817096;C1306645,C0817096;C1306645
2,ImageCLEFmedical_Caption_2026_train_2,train,CT chest axial view showing a huge ascending a...,C0040405;C0856747,C0040405,C0040405
3,ImageCLEFmedical_Caption_2026_train_3,train,Acquired renal cysts in end-stage renal failur...,C0041618,C0041618,C0041618
4,ImageCLEFmedical_Caption_2026_train_4,train,Computed tomography (CT) shows floating thromb...,C0040405;C0040053,C0040405,C0040405


In [7]:
if not MAPPING_PATH.exists():
    raise FileNotFoundError(f"Mapping file not found:\n{MAPPING_PATH}")

mapping_df = pd.read_csv(MAPPING_PATH)

print("Mapping shape:", mapping_df.shape)
print("Mapping columns:")
print(mapping_df.columns.tolist())

display(mapping_df.head())


def detect_cui_column(df):
    common_names = ["cui", "CUI", "umls_cui", "UMLS_CUI", "concept_id", "Concept_ID"]

    for name in common_names:
        if name in df.columns:
            return name

    pattern = re.compile(r"^C\d{7}$")
    best_col = None
    best_count = 0

    for col in df.columns:
        values = df[col].dropna().astype(str).head(300)
        count = sum(bool(pattern.match(v.strip())) for v in values)

        if count > best_count:
            best_count = count
            best_col = col

    if best_col is None:
        raise ValueError("Could not detect CUI column.")

    return best_col


def detect_term_column(df, cui_col):
    common_names = [
        "term", "Term",
        "name", "Name",
        "preferred_name", "Preferred_Name",
        "preferred_term", "Preferred_Term",
        "umls_term", "UMLS_Term",
        "label", "Label",
        "STR", "str"
    ]

    for name in common_names:
        if name in df.columns and name != cui_col:
            return name

    candidate_cols = [c for c in df.columns if c != cui_col]

    best_col = None
    best_score = -1

    for col in candidate_cols:
        values = df[col].dropna().astype(str).head(300)
        if len(values) == 0:
            continue

        avg_len = values.str.len().mean()
        alpha_count = values.str.contains(r"[A-Za-z]", regex=True).sum()
        score = avg_len + alpha_count

        if score > best_score:
            best_score = score
            best_col = col

    if best_col is None:
        raise ValueError("Could not detect UMLS term column.")

    return best_col


cui_col = detect_cui_column(mapping_df)
term_col = detect_term_column(mapping_df, cui_col)

print("Detected CUI column:", cui_col)
print("Detected term column:", term_col)

mapping_df[cui_col] = mapping_df[cui_col].astype(str).str.strip()
mapping_df[term_col] = mapping_df[term_col].astype(str).str.strip()

mapping_df = mapping_df[
    mapping_df[cui_col].str.match(r"^C\d{7}$", na=False)
].copy()

mapping_df = mapping_df[
    mapping_df[term_col].notna()
    & (mapping_df[term_col].str.strip() != "")
    & (mapping_df[term_col].str.lower().str.strip() != "nan")
].copy()

mapping_df = mapping_df.drop_duplicates(subset=[cui_col], keep="first")

cui_to_term = dict(zip(mapping_df[cui_col], mapping_df[term_col]))

cui_to_term["C0043262"] = "Wrist"

mapping_path = MAPPING_PATH

print("\nLoaded mappings:", len(cui_to_term))
print("Example C0040405:", cui_to_term.get("C0040405"))
print("Example C0043262:", cui_to_term.get("C0043262"))

missing_in_mapping = [c for c in cui_list if c not in cui_to_term]
print("CUIs missing from mapping:", len(missing_in_mapping))
print("First missing:", missing_in_mapping[:20])

Mapping shape: (2646, 5)
Mapping columns:
['CUI', 'name', 'semantic_types', 'status_code', 'error']


,CUI,name,semantic_types,status_code,error
0,C0000726,Abdomen,Body Location or Region,200,NaN
1,C0000741,Abducens nerve structure,"Body Part, Organ, or Organ Component",200,NaN
2,C0000833,Abscess,Disease or Syndrome,200,NaN
3,C0000846,Agenesis,Congenital Abnormality,200,NaN
4,C0000962,Bone structure of acetabulum,"Body Part, Organ, or Organ Component",200,NaN


Detected CUI column: CUI
Detected term column: name

Loaded mappings: 2639
Example C0040405: X-Ray Computed Tomography
Example C0043262: Wrist
CUIs missing from mapping: 7
First missing: ['C0206702', 'C0227130', 'C0241790', 'C0555895', 'C1134719', 'C1321581', 'C1458136']


In [8]:
def clean_cuis(x):
    if pd.isna(x):
        return []

    s = str(x).strip()

    if s == "" or s.lower() in ["nan", "none", "null"]:
        return []

    s = s.replace(",", ";")
    parts = []

    for chunk in s.split(";"):
        chunk = chunk.strip()
        if not chunk:
            continue

        for sp in chunk.split():
            sp = sp.strip()
            if sp:
                parts.append(sp)

    seen = set()
    clean = []

    for c in parts:
        if c not in seen:
            seen.add(c)
            clean.append(c)

    return clean


def cui_to_readable_piece(cui, mapping):
    term = mapping.get(cui, None)

    if term is None or str(term).strip() == "":
        return f"<{cui}>"

    return f"<{cui}> {str(term).strip()}"


def make_cui_terms_input(cui_string, source_name, mapping):
    cuis = clean_cuis(cui_string)

    if len(cuis) == 0:
        concept_text = "<NO_CONCEPTS>"
    else:
        pieces = [cui_to_readable_piece(cui, mapping) for cui in cuis]
        concept_text = "; ".join(pieces)

    return f"generate medical caption from {source_name} umls concepts: {concept_text}"


row0 = master_df.iloc[0]

print("GT example:")
print(make_cui_terms_input(row0["gt_CUIs"], "ground-truth", cui_to_term))

print("\nPred micro example:")
print(make_cui_terms_input(row0["pred_CUIs_micro"], "predicted", cui_to_term))

GT example:
generate medical caption from ground-truth umls concepts: <C0040405> X-Ray Computed Tomography

Pred micro example:
generate medical caption from predicted umls concepts: <C0040405> X-Ray Computed Tomography


In [9]:
df = master_df.copy()

df["reference_caption"] = df["reference_caption"].astype(str)

df = df[
    df["reference_caption"].notna()
    & (df["reference_caption"].str.strip() != "")
    & (df["reference_caption"].str.lower().str.strip() != "nan")
].reset_index(drop=True)

train_base_df = df[df["split"] == "train"].copy().reset_index(drop=True)
valid_base_df = df[df["split"] == "valid"].copy().reset_index(drop=True)

if len(train_base_df) == 0:
    raise ValueError("train_base_df is empty.")

if len(valid_base_df) == 0:
    raise ValueError("valid_base_df is empty.")

rng = np.random.default_rng(SEED)

train_use_gt = rng.random(len(train_base_df)) < MIX_GT_RATIO

train_base_df["input_source"] = np.where(train_use_gt, "gt", "pred_micro")

train_base_df["input_text"] = np.where(
    train_base_df["input_source"] == "gt",
    train_base_df["gt_CUIs"].apply(lambda x: make_cui_terms_input(x, "ground-truth", cui_to_term)),
    train_base_df["pred_CUIs_micro"].apply(lambda x: make_cui_terms_input(x, "predicted", cui_to_term))
)

train_base_df["target_text"] = train_base_df["reference_caption"].astype(str).str.strip()

train_full_df = train_base_df[[
    "ID",
    "input_source",
    "input_text",
    "target_text"
]].reset_index(drop=True)

valid_n = min(VALID_SAMPLE_SIZE, len(valid_base_df))
valid_sample_df = valid_base_df.sample(n=valid_n, random_state=SEED).reset_index(drop=True)

valid_rng = np.random.default_rng(SEED + 1)
valid_use_gt = valid_rng.random(len(valid_sample_df)) < MIX_GT_RATIO

valid_sample_df["input_source"] = np.where(valid_use_gt, "gt", "pred_micro")

valid_sample_df["input_text"] = np.where(
    valid_sample_df["input_source"] == "gt",
    valid_sample_df["gt_CUIs"].apply(lambda x: make_cui_terms_input(x, "ground-truth", cui_to_term)),
    valid_sample_df["pred_CUIs_micro"].apply(lambda x: make_cui_terms_input(x, "predicted", cui_to_term))
)

valid_sample_df["target_text"] = valid_sample_df["reference_caption"].astype(str).str.strip()

valid_eval_df = valid_sample_df[[
    "ID",
    "input_source",
    "input_text",
    "target_text"
]].reset_index(drop=True)

if FAST_DEBUG:
    train_n = min(TRAIN_DEBUG_SIZE, len(train_full_df))
    valid_debug_n = min(VALID_DEBUG_SIZE, len(valid_eval_df))

    train_full_df = train_full_df.sample(n=train_n, random_state=SEED).reset_index(drop=True)
    valid_eval_df = valid_eval_df.sample(n=valid_debug_n, random_state=SEED).reset_index(drop=True)

print("Train full:", train_full_df.shape)
print("Valid eval:", valid_eval_df.shape)

print("\nTrain input source counts:")
print(train_full_df["input_source"].value_counts())

print("\nValid input source counts:")
print(valid_eval_df["input_source"].value_counts())

print("\nExample input:")
print(train_full_df.iloc[0]["input_text"])

print("\nExample target:")
print(train_full_df.iloc[0]["target_text"])

Train full: (97364, 4)
Valid eval: (3000, 4)

Train input source counts:
input_source
pred_micro    48932
gt            48432
Name: count, dtype: int64

Valid input source counts:
input_source
gt            1517
pred_micro    1483
Name: count, dtype: int64

Example input:
generate medical caption from predicted umls concepts: <C0040405> X-Ray Computed Tomography

Example target:
Head CT demonstrating left parotiditis.


In [10]:
manual_candidates = [
    PROJECT_DIR / "models" / "gt_cui_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "models" / "gt_cui_umls_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "models" / "gt_umls_terms_t5_base_full" / "checkpoint-3043",
    PROJECT_DIR / "models" / "cui_terms_t5_base_full" / "checkpoint-3043",
]

auto_candidates = []
for root in [PROJECT_DIR, PROJECT_DIR / "models", PROJECT_DIR / "caption_generator_datasets_umls_terms"]:
    if root.exists():
        auto_candidates.extend(list(root.rglob("checkpoint-3043")))

all_candidates = manual_candidates + auto_candidates

valid_model_candidates = []
for p in all_candidates:
    if p.exists() and (p / "config.json").exists():
        valid_model_candidates.append(p)

BASE_CHECKPOINT = None

preferred = [
    p for p in valid_model_candidates
    if "gt_cui_terms_t5_base_full" in str(p)
]

if len(preferred) > 0:
    BASE_CHECKPOINT = preferred[0]
elif len(valid_model_candidates) > 0:
    BASE_CHECKPOINT = valid_model_candidates[0]

print("Checkpoint candidates:")
for p in valid_model_candidates[:20]:
    print(" -", p)

if BASE_CHECKPOINT is not None:
    model_source = str(BASE_CHECKPOINT)
    print("\nStarting from checkpoint:", model_source)
else:
    model_source = MODEL_NAME
    print("\nNo checkpoint found. Starting from:", MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

special_tokens = [f"<{cui}>" for cui in cui_list] + ["<NO_CONCEPTS>"]
num_added = tokenizer.add_tokens(special_tokens)

print("Added special tokens:", num_added)
print("Tokenizer size:", len(tokenizer))

model = AutoModelForSeq2SeqLM.from_pretrained(model_source)

old_vocab_size = model.get_input_embeddings().weight.shape[0]
new_vocab_size = len(tokenizer)

if old_vocab_size != new_vocab_size:
    print(f"Resizing token embeddings: {old_vocab_size} -> {new_vocab_size}")
    model.resize_token_embeddings(new_vocab_size)
else:
    print("Embedding size already matches tokenizer size.")

model.config.use_cache = False

print("Model loaded successfully.")

Checkpoint candidates:
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/checkpoint-3043
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043
 - /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp3_pred_micro_terms_t5_base_safe5/checkpoint-3043

Starting from checkpoint: /content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Added special tokens: 2647
Tokenizer size: 34747


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Resizing token embeddings: 32128 -> 34747
Model loaded successfully.


In [11]:
MAX_INPUT_LEN = 384
MAX_TARGET_LEN = 96

train_dataset = Dataset.from_pandas(train_full_df)
valid_dataset = Dataset.from_pandas(valid_eval_df)

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_INPUT_LEN,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_TARGET_LEN,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

print(tokenized_train)
print(tokenized_valid)

sample_item = tokenized_train[0]
print("\nInput length:", len(sample_item["input_ids"]))
print("Label length:", len(sample_item["labels"]))

Map:   0%|          | 0/97364 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 97364
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})

Input length: 21
Label length: 14


In [12]:
import evaluate

rouge = evaluate.load("rouge")
bleu = evaluate.load("sacrebleu")

def compute_metrics(eval_pred):
    preds, labels = eval_pred

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.asarray(preds)
    labels = np.asarray(labels)

    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    vocab_size = len(tokenizer)

    preds = np.where(
        (preds < 0) | (preds >= vocab_size),
        tokenizer.pad_token_id,
        preds
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id
    )

    preds = preds.astype(np.int64)
    labels = labels.astype(np.int64)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    rouge_result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    bleu_result = bleu.compute(
        predictions=decoded_preds,
        references=[[l] for l in decoded_labels]
    )

    return {
        "rouge1": rouge_result["rouge1"],
        "rouge2": rouge_result["rouge2"],
        "rougeL": rouge_result["rougeL"],
        "bleu": bleu_result["score"]
    }

In [13]:
from transformers import DataCollatorForSeq2Seq

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

mini_features = [tokenized_train[i] for i in range(min(2, len(tokenized_train)))]
mini_batch = data_collator(mini_features)

mini_batch = {
    k: v.to(device) if hasattr(v, "to") else v
    for k, v in mini_batch.items()
}

with torch.no_grad():
    outputs = model(**mini_batch)
    test_loss = outputs.loss

print("Safety test loss:", float(test_loss.detach().cpu()))

if not torch.isfinite(test_loss):
    raise ValueError("Non-finite loss detected before training.")
else:
    print("Safety check passed.")

Safety test loss: 3.2226991653442383
Safety check passed.


In [14]:
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
    TrainerCallback,
    DataCollatorForSeq2Seq
)
from transformers.trainer_utils import get_last_checkpoint

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = False

print("use_bf16:", use_bf16)
print("use_fp16:", use_fp16)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

model.config.use_cache = False


class StopOnNonFiniteCallback(TrainerCallback):
    def _check_values(self, values, control, stage):
        if values is None:
            return control

        for key, value in values.items():
            if isinstance(value, (int, float, np.floating)):
                if not np.isfinite(value):
                    print(f"\n[SAFETY STOP] Non-finite value detected during {stage}: {key}={value}")
                    control.should_save = True
                    control.should_training_stop = True
                    return control

        return control

    def on_log(self, args, state, control, logs=None, **kwargs):
        return self._check_values(logs, control, "logging")

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        return self._check_values(metrics, control, "evaluation")


args_dict = dict(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=5,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=2e-5,
    warmup_ratio=0.08,
    lr_scheduler_type="linear",

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,

    max_grad_norm=0.5,
    weight_decay=0.01,
    adam_epsilon=1e-6,
    label_smoothing_factor=0.05,

    bf16=use_bf16,
    fp16=use_fp16,

    predict_with_generate=True,
    generation_max_length=96,
    generation_num_beams=2,

    eval_accumulation_steps=8,

    logging_steps=100,
    save_total_limit=3,
    report_to="none",
    logging_nan_inf_filter=True,

    load_best_model_at_end=True,
    metric_for_best_model="eval_rougeL",
    greater_is_better=True,

    gradient_checkpointing=True,

    seed=SEED,
    data_seed=SEED,
)

supported_args = set(inspect.signature(Seq2SeqTrainingArguments.__init__).parameters.keys())

if "eval_strategy" not in supported_args and "evaluation_strategy" in supported_args:
    args_dict["evaluation_strategy"] = args_dict.pop("eval_strategy")

filtered_args = {}
removed_args = []

for key, value in args_dict.items():
    if key in supported_args:
        filtered_args[key] = value
    else:
        removed_args.append(key)

print("\nUnsupported TrainingArguments removed:")
print(removed_args)

training_args = Seq2SeqTrainingArguments(**filtered_args)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        StopOnNonFiniteCallback(),
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0005
        )
    ]
)

last_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
print("\nLast checkpoint:", last_checkpoint)

if last_checkpoint is not None:
    print("Resuming from checkpoint:", last_checkpoint)
    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    print("Starting fresh Exp4 mixed training.")
    train_result = trainer.train()

BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
trainer.save_model(str(BEST_MODEL_DIR))
tokenizer.save_pretrained(str(BEST_MODEL_DIR))

print("\nSaved best model to:", BEST_MODEL_DIR)

train_metrics = train_result.metrics

trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)
trainer.save_state()

history_df = pd.DataFrame(trainer.state.log_history)
history_path = OUTPUT_DIR / "training_log_history.csv"
history_df.to_csv(history_path, index=False)

print("\nTraining completed.")
print(train_metrics)
print("Saved history to:", history_path)

display(history_df.tail(20))

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


use_bf16: True
use_fp16: False
GPU: NVIDIA A100-SXM4-80GB

Unsupported TrainingArguments removed:
[]

Last checkpoint: None
Starting fresh Exp4 mixed training.


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Bleu
1,13.526267,3.201525,0.275997,0.106421,0.241758,4.140052
2,13.083351,3.136454,0.265682,0.101109,0.231565,4.149699
3,13.057537,3.104669,0.269102,0.104741,0.233909,4.235105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved best model to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5/best_model
***** train metrics *****
  epoch                    =        3.0
  total_flos               = 15614341GF
  train_loss               =    13.4855
  train_runtime            = 3:05:49.77
  train_samples_per_second =     43.662
  train_steps_per_second   =      1.365

Training completed.
{'train_runtime': 11149.7798, 'train_samples_per_second': 43.662, 'train_steps_per_second': 1.365, 'total_flos': 1.676577155309568e+16, 'train_loss': 13.485518160596554, 'epoch': 3.0}
Saved history to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5/training_log_history.csv


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_rouge1,eval_rouge2,eval_rougeL,eval_bleu,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
75,13.082390,3.011845,0.000011,2.431846,7400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,12.946730,2.584981,0.000011,2.464711,7500,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,13.089609,2.981140,0.000011,2.497576,7600,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,13.018091,2.647464,0.000011,2.530441,7700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,13.022639,2.836571,0.000011,2.563306,7800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,13.030493,3.139159,0.000010,2.596171,7900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,12.948319,3.294915,0.000010,2.629036,8000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,13.048746,3.012743,0.000010,2.661901,8100,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,12.982584,2.714570,0.000010,2.694766,8200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,12.906462,2.942230,0.000010,2.727631,8300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
valid_ids = valid_eval_df["ID"].astype(str).tolist()

eval_base_df = master_df[master_df["ID"].astype(str).isin(valid_ids)].copy()

eval_base_df["ID_order"] = pd.Categorical(
    eval_base_df["ID"].astype(str),
    categories=valid_ids,
    ordered=True
)

eval_base_df = eval_base_df.sort_values("ID_order").drop(columns=["ID_order"]).reset_index(drop=True)

print("Eval base shape:", eval_base_df.shape)
print("Expected valid IDs:", len(valid_ids))


def build_eval_variant(base_df, cui_col, source_name):
    temp = base_df.copy()

    temp["input_text"] = temp[cui_col].apply(
        lambda x: make_cui_terms_input(x, source_name, cui_to_term)
    )

    temp["target_text"] = temp["reference_caption"].astype(str).str.strip()

    return temp[["ID", "input_text", "target_text"]].reset_index(drop=True)


valid_gt_eval = build_eval_variant(eval_base_df, "gt_CUIs", "ground-truth")
valid_micro_eval = build_eval_variant(eval_base_df, "pred_CUIs_micro", "predicted")
valid_coverage_eval = build_eval_variant(eval_base_df, "pred_CUIs_coverage", "predicted")

print("GT eval:", valid_gt_eval.shape)
print("Pred micro eval:", valid_micro_eval.shape)
print("Pred coverage eval:", valid_coverage_eval.shape)

print("\nGT example:")
print(valid_gt_eval.iloc[0]["input_text"])

print("\nPred micro example:")
print(valid_micro_eval.iloc[0]["input_text"])

Eval base shape: (3000, 20)
Expected valid IDs: 3000
GT eval: (3000, 3)
Pred micro eval: (3000, 3)
Pred coverage eval: (3000, 3)

GT example:
generate medical caption from ground-truth umls concepts: <C0041618> Ultrasonography; <C0182400> Probes; <C0042449> Veins

Pred micro example:
generate medical caption from predicted umls concepts: <C0041618> Ultrasonography


In [16]:
def tokenize_eval_df(eval_df):
    dataset = Dataset.from_pandas(eval_df)

    tokenized = dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=dataset.column_names
    )

    return tokenized


tokenized_valid_gt = tokenize_eval_df(valid_gt_eval)
tokenized_valid_micro = tokenize_eval_df(valid_micro_eval)
tokenized_valid_coverage = tokenize_eval_df(valid_coverage_eval)

print(tokenized_valid_gt)
print(tokenized_valid_micro)
print(tokenized_valid_coverage)

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})


In [17]:
trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if not isinstance(cb, EarlyStoppingCallback)
]

gt_metrics = trainer.evaluate(
    eval_dataset=tokenized_valid_gt,
    metric_key_prefix="gt"
)

micro_metrics = trainer.evaluate(
    eval_dataset=tokenized_valid_micro,
    metric_key_prefix="pred_micro"
)

coverage_metrics = trainer.evaluate(
    eval_dataset=tokenized_valid_coverage,
    metric_key_prefix="pred_coverage"
)

comparison = pd.DataFrame([
    {
        "model": "Exp4 Mixed GT+Pred-micro T5-base",
        "setting": "GT concepts + UMLS terms",
        "loss": gt_metrics.get("gt_loss"),
        "rouge1": gt_metrics.get("gt_rouge1"),
        "rouge2": gt_metrics.get("gt_rouge2"),
        "rougeL": gt_metrics.get("gt_rougeL"),
        "bleu": gt_metrics.get("gt_bleu"),
    },
    {
        "model": "Exp4 Mixed GT+Pred-micro T5-base",
        "setting": "Pred micro concepts + UMLS terms",
        "loss": micro_metrics.get("pred_micro_loss"),
        "rouge1": micro_metrics.get("pred_micro_rouge1"),
        "rouge2": micro_metrics.get("pred_micro_rouge2"),
        "rougeL": micro_metrics.get("pred_micro_rougeL"),
        "bleu": micro_metrics.get("pred_micro_bleu"),
    },
    {
        "model": "Exp4 Mixed GT+Pred-micro T5-base",
        "setting": "Pred coverage concepts + UMLS terms",
        "loss": coverage_metrics.get("pred_coverage_loss"),
        "rouge1": coverage_metrics.get("pred_coverage_rouge1"),
        "rouge2": coverage_metrics.get("pred_coverage_rouge2"),
        "rougeL": coverage_metrics.get("pred_coverage_rougeL"),
        "bleu": coverage_metrics.get("pred_coverage_bleu"),
    },
])

display(comparison)

comparison_path = OUTPUT_DIR / "exp4_mixed_gt_pred_eval_comparison_valid3000.csv"
comparison.to_csv(comparison_path, index=False)

print("Saved comparison to:", comparison_path)

Training Loss,Validation Loss,Epoch,Rouge1,Rouge2,Rougel,Bleu
13.057537,3.044808,3,0.332843,0.141381,0.294864,5.657391


Training Loss,Validation Loss,Epoch,Micro Loss,Micro Rouge1,Micro Rouge2,Micro Rougel,Micro Bleu
13.057537,No log,3,3.360227,0.217945,0.070818,0.188926,2.594652


Training Loss,Validation Loss,Epoch,Coverage Loss,Coverage Rouge1,Coverage Rouge2,Coverage Rougel,Coverage Bleu
13.057537,No log,3,3.358000,0.218546,0.071832,0.189702,2.528929


,model,setting,loss,rouge1,rouge2,rougeL,bleu
0,Exp4 Mixed GT+Pred-micro T5-base,GT concepts + UMLS terms,3.044808,0.332843,0.141381,0.294864,5.657391
1,Exp4 Mixed GT+Pred-micro T5-base,Pred micro concepts + UMLS terms,3.360227,0.217945,0.070818,0.188926,2.594652
2,Exp4 Mixed GT+Pred-micro T5-base,Pred coverage concepts + UMLS terms,3.358000,0.218546,0.071832,0.189702,2.528929


Saved comparison to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5/exp4_mixed_gt_pred_eval_comparison_valid3000.csv


In [18]:
previous_results = pd.DataFrame([
    {
        "model": "Exp2 GT-trained",
        "setting": "GT concepts + UMLS terms",
        "loss": 2.911293,
        "rouge1": 0.276256,
        "rouge2": 0.112285,
        "rougeL": 0.245186,
        "bleu": 5.134103,
    },
    {
        "model": "Exp2 GT-trained",
        "setting": "Pred micro concepts + UMLS terms",
        "loss": 3.370372,
        "rouge1": 0.145410,
        "rouge2": 0.043019,
        "rougeL": 0.130601,
        "bleu": 1.209188,
    },
    {
        "model": "Exp2 GT-trained",
        "setting": "Pred coverage concepts + UMLS terms",
        "loss": 3.369131,
        "rouge1": 0.144617,
        "rouge2": 0.044812,
        "rougeL": 0.129142,
        "bleu": 1.220970,
    },
    {
        "model": "Exp3 Pred-micro fine-tuned safe5",
        "setting": "GT concepts + UMLS terms",
        "loss": 3.113310,
        "rouge1": 0.288020,
        "rouge2": 0.106795,
        "rougeL": 0.247639,
        "bleu": 5.020259,
    },
    {
        "model": "Exp3 Pred-micro fine-tuned safe5",
        "setting": "Pred micro concepts + UMLS terms",
        "loss": 3.340528,
        "rouge1": 0.208327,
        "rouge2": 0.063280,
        "rougeL": 0.180706,
        "bleu": 2.276206,
    },
    {
        "model": "Exp3 Pred-micro fine-tuned safe5",
        "setting": "Pred coverage concepts + UMLS terms",
        "loss": 3.337225,
        "rouge1": 0.211430,
        "rouge2": 0.065665,
        "rougeL": 0.183133,
        "bleu": 2.299728,
    },
])

full_comparison = pd.concat(
    [previous_results, comparison],
    ignore_index=True
)

old_pred_micro_rougeL = 0.130601
old_pred_micro_bleu = 1.209188

exp3_pred_micro_rougeL = 0.180706
exp3_pred_micro_bleu = 2.276206

full_comparison["delta_vs_exp2_pred_micro_rougeL"] = full_comparison["rougeL"] - old_pred_micro_rougeL
full_comparison["delta_vs_exp2_pred_micro_bleu"] = full_comparison["bleu"] - old_pred_micro_bleu

full_comparison["delta_vs_exp3_pred_micro_rougeL"] = full_comparison["rougeL"] - exp3_pred_micro_rougeL
full_comparison["delta_vs_exp3_pred_micro_bleu"] = full_comparison["bleu"] - exp3_pred_micro_bleu

display(full_comparison)

full_comparison_path = OUTPUT_DIR / "exp2_exp3_exp4_full_comparison_valid3000.csv"
full_comparison.to_csv(full_comparison_path, index=False)

print("Saved full comparison to:", full_comparison_path)

,model,setting,loss,rouge1,rouge2,rougeL,bleu,delta_vs_exp2_pred_micro_rougeL,delta_vs_exp2_pred_micro_bleu,delta_vs_exp3_pred_micro_rougeL,delta_vs_exp3_pred_micro_bleu
0,Exp2 GT-trained,GT concepts + UMLS terms,2.911293,0.276256,0.112285,0.245186,5.134103,0.114585,3.924915,0.064480,2.857897
1,Exp2 GT-trained,Pred micro concepts + UMLS terms,3.370372,0.145410,0.043019,0.130601,1.209188,0.000000,0.000000,-0.050105,-1.067018
2,Exp2 GT-trained,Pred coverage concepts + UMLS terms,3.369131,0.144617,0.044812,0.129142,1.220970,-0.001459,0.011782,-0.051564,-1.055236
3,Exp3 Pred-micro fine-tuned safe5,GT concepts + UMLS terms,3.113310,0.288020,0.106795,0.247639,5.020259,0.117038,3.811071,0.066933,2.744053
4,Exp3 Pred-micro fine-tuned safe5,Pred micro concepts + UMLS terms,3.340528,0.208327,0.063280,0.180706,2.276206,0.050105,1.067018,0.000000,0.000000
5,Exp3 Pred-micro fine-tuned safe5,Pred coverage concepts + UMLS terms,3.337225,0.211430,0.065665,0.183133,2.299728,0.052532,1.090540,0.002427,0.023522
6,Exp4 Mixed GT+Pred-micro T5-base,GT concepts + UMLS terms,3.044808,0.332843,0.141381,0.294864,5.657391,0.164263,4.448203,0.114158,3.381185
7,Exp4 Mixed GT+Pred-micro T5-base,Pred micro concepts + UMLS terms,3.360227,0.217945,0.070818,0.188926,2.594652,0.058325,1.385464,0.008220,0.318446
8,Exp4 Mixed GT+Pred-micro T5-base,Pred coverage concepts + UMLS terms,3.358000,0.218546,0.071832,0.189702,2.528929,0.059101,1.319741,0.008996,0.252723


Saved full comparison to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5/exp2_exp3_exp4_full_comparison_valid3000.csv


In [19]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model.to(device)
model.eval()

sample_n = min(10, len(eval_base_df))
sample_df = eval_base_df.sample(n=sample_n, random_state=11).reset_index(drop=True)

qualitative_rows = []

def generate_caption_from_input(input_text, num_beams=4):
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=96,
            num_beams=num_beams,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()


for _, row in sample_df.iterrows():
    sample_id = row["ID"]
    target = row["reference_caption"]

    gt_input = make_cui_terms_input(row["gt_CUIs"], "ground-truth", cui_to_term)
    micro_input = make_cui_terms_input(row["pred_CUIs_micro"], "predicted", cui_to_term)
    coverage_input = make_cui_terms_input(row["pred_CUIs_coverage"], "predicted", cui_to_term)

    gt_pred = generate_caption_from_input(gt_input)
    micro_pred = generate_caption_from_input(micro_input)
    coverage_pred = generate_caption_from_input(coverage_input)

    qualitative_rows.append({
        "ID": sample_id,
        "target": target,
        "gt_CUIs": row["gt_CUIs"],
        "pred_CUIs_micro": row["pred_CUIs_micro"],
        "pred_CUIs_coverage": row["pred_CUIs_coverage"],
        "gt_prediction": gt_pred,
        "micro_prediction": micro_pred,
        "coverage_prediction": coverage_pred,
    })

    print("=" * 120)
    print("ID:", sample_id)

    print("\nTARGET:")
    print(target)

    print("\nGT PREDICTION:")
    print(gt_pred)

    print("\nMICRO PREDICTION:")
    print(micro_pred)

    print("\nCOVERAGE PREDICTION:")
    print(coverage_pred)

qualitative_df = pd.DataFrame(qualitative_rows)

qualitative_path = OUTPUT_DIR / "exp4_mixed_qualitative_samples.csv"
qualitative_df.to_csv(qualitative_path, index=False)

print("\nSaved qualitative samples to:", qualitative_path)

display(qualitative_df[[
    "ID",
    "target",
    "gt_prediction",
    "micro_prediction",
    "coverage_prediction"
]])

ID: ImageCLEFmedical_Caption_2026_valid_4390

TARGET:
Example of an anterior–posterior radiograph of the pelvis with calculation of the migration percentage (MP): MP = A/B × 100. A represents the portion of ossified femoral head laying lateral to Perkin's line (vertical line drawn through the lateral acetabular margin and perpendicular to Hilgenreiner's line, which passes through the superior aspect of the triradiate cartilage). B represents the whole ossified femoral head.

GT PREDICTION:
Anteroposterior radiograph of the pelvis showing the femoral head and cartilage.

MICRO PREDICTION:
Anteroposterior radiograph of the pelvis showing a large obstructive lesion in the right lower lobe of the pelvis.

COVERAGE PREDICTION:
Anteroposterior radiograph of the femoral head.
ID: ImageCLEFmedical_Caption_2026_valid_15145

TARGET:
Coronal view of contrast-enhanced magnetic resonance imaging (MRI) showing an inhomogeneously enhancing mass within the proximal great saphenous vein (GSV) expanding

,ID,target,gt_prediction,micro_prediction,coverage_prediction
0,ImageCLEFmedical_Caption_2026_valid_4390,Example of an anterior–posterior radiograph of...,Anteroposterior radiograph of the pelvis showi...,Anteroposterior radiograph of the pelvis showi...,Anteroposterior radiograph of the femoral head.
1,ImageCLEFmedical_Caption_2026_valid_15145,Coronal view of contrast-enhanced magnetic res...,Sagittal T1-weighted MRI of the saphenous vein...,Axial T1-weighted MRI of the thoracic spine de...,Axial T1-weighted MRI of the thoracic spine de...
2,ImageCLEFmedical_Caption_2026_valid_5073,Radiographic check-up after the provisional re...,Panoramic radiograph of the patient.,Panoramic radiograph of the patient.,Panoramic radiograph of the patient.
3,ImageCLEFmedical_Caption_2026_valid_2299,A 57-year-old woman underwent pelvic ultrasoun...,Transesophageal echocardiogram of the pelvis.,Transesophageal echocardiography showing a lar...,Transesophageal echocardiography showing a lar...
4,ImageCLEFmedical_Caption_2026_valid_15195,Image showing right-sided bipolar hemiarthropl...,Anteroposterior X-ray of the right knee.,Anteroposterior X-ray of the right knee showin...,Anteroposterior X-ray of the right knee showin...
5,ImageCLEFmedical_Caption_2026_valid_19195,Coronal T2 weighted magnetic resonance image o...,Magnetic resonance imaging of the left ankle.,Axial T1-weighted MRI of the thoracic spine de...,Axial T1-weighted MRI of the thoracic spine de...
6,ImageCLEFmedical_Caption_2026_valid_265,Pneumomediastinum in the superior mediastinum ...,Computed tomography scan showing pneumomediast...,Computed tomography (CT) scan of the abdomen a...,Chest computed tomography (CT) scan showing a ...
7,ImageCLEFmedical_Caption_2026_valid_11915,Day 2 transvaginal sonography of the patient's...,Ultrasound image of the right ovary.,Transesophageal echocardiography showing a lar...,Transesophageal echocardiography showing a lar...
8,ImageCLEFmedical_Caption_2026_valid_9689,Repeat chest X-ray after one week. In comparis...,Chest X-ray showing pneumomediastinum.,Chest X-ray showing a large opacity in the rig...,Chest X-ray showing a large opacity in the rig...
9,ImageCLEFmedical_Caption_2026_valid_10849,Post EM Titration (admit 2).Marked left lower ...,Chest X-ray showing left lower lobe atelectasi...,Chest X-ray showing a large opacity in the rig...,Chest X-ray showing a large opacity in the rig...


In [20]:
run_summary = {
    "experiment": "Experiment 4 - Mixed GT and Predicted UMLS Concept Training",
    "variant": "mixed_gt_pred_micro_terms_t5_base_safe5",
    "project_dir": str(PROJECT_DIR),
    "master_path": str(MASTER_PATH),
    "mapping_path": str(MAPPING_PATH),
    "label_vocab_path": str(LABEL_VOCAB_PATH),
    "output_dir": str(OUTPUT_DIR),
    "base_model_or_checkpoint": model_source,
    "model_name": MODEL_NAME,
    "num_cuis": len(cui_list),
    "train_size": int(len(train_full_df)),
    "valid_eval_size": int(len(valid_eval_df)),
    "mix_gt_ratio": MIX_GT_RATIO,
    "mix_pred_micro_ratio": 1.0 - MIX_GT_RATIO,
    "max_input_len": MAX_INPUT_LEN,
    "max_target_len": MAX_TARGET_LEN,
    "training_epochs_max": 5,
    "learning_rate": 2e-5,
    "warmup_ratio": 0.08,
    "effective_batch_size": 8 * 4,
    "precision": "bf16" if use_bf16 else "fp32",
    "fp16_used": False,
    "best_model_dir": str(BEST_MODEL_DIR),
    "comparison_csv": str(comparison_path),
    "full_comparison_csv": str(full_comparison_path),
    "qualitative_csv": str(qualitative_path),
}

summary_path = OUTPUT_DIR / "exp4_mixed_run_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2, ensure_ascii=False)

print("Saved run summary to:", summary_path)
print(json.dumps(run_summary, indent=2, ensure_ascii=False))

Saved run summary to: /content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5/exp4_mixed_run_summary.json
{
  "experiment": "Experiment 4 - Mixed GT and Predicted UMLS Concept Training",
  "variant": "mixed_gt_pred_micro_terms_t5_base_safe5",
  "project_dir": "/content/drive/MyDrive/Senior2_Medical_Captioning",
  "master_path": "/content/drive/MyDrive/Senior2_Medical_Captioning/master_caption_concepts_scored.csv",
  "mapping_path": "/content/drive/MyDrive/Senior2_Medical_Captioning/cui_to_umls_terms.csv",
  "label_vocab_path": "/content/drive/MyDrive/ImageCLEF/artifacts_consultive_swin/label_vocab.json",
  "output_dir": "/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5",
  "base_model_or_checkpoint": "/content/drive/MyDrive/Senior2_Medical_Captioning/models/gt_cui_terms_t5_base_full/checkpoint-3043",
  "model_name": "t5-base",
  "num_cuis": 2646,
  "train_size": 97364,
  "valid_eval_size"

In [21]:
from pathlib import Path

exp4_dir = Path("/content/drive/MyDrive/Senior2_Medical_Captioning/models/exp4_mixed_gt_pred_micro_terms_t5_base_safe5")

for p in exp4_dir.iterdir():
    print(p.name)

checkpoint-3043
checkpoint-6086
checkpoint-9129
best_model
train_results.json
all_results.json
trainer_state.json
training_log_history.csv
exp4_mixed_gt_pred_eval_comparison_valid3000.csv
exp2_exp3_exp4_full_comparison_valid3000.csv
exp4_mixed_qualitative_samples.csv
exp4_mixed_run_summary.json
